# VoiceChess-ID — Persiapan & Evaluasi Dataset

Urutan: **1** mount Drive → **2** tulis skrip → **3** parser → **4** satukan dataset → **5** cek hasil potongan & label → **6** dry-run parser → **7** evaluasi Whisper.

**Sebelum mulai:**
- Di Drive: klik kanan folder *Project KLP 5 - SR* → *Organize → Add shortcut* → **My Drive**.
- Runtime → Change runtime type → **T4 GPU** (untuk langkah 7). Latency final tetap diukur di laptop demo.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SRC = "/content/drive/MyDrive/Project KLP 5 - SR/Rekaman Data"   # ganti kalau shortcut-nya di tempat lain
import os
print('Folder ditemukan' if os.path.isdir(SRC) else 'FOLDER TIDAK DITEMUKAN: cek path / shortcut')
print(sorted(os.listdir(SRC)) if os.path.isdir(SRC) else '')

## 2. Tulis skrip
Dua sel ini hanya menyimpan file `.py` ke Colab (tidak menjalankan apa-apa).

In [ ]:
%%writefile prepare_dataset.py
"""
VoiceChess-ID - Satukan semua rekaman di folder Drive "Rekaman Data" menjadi satu
dataset seragam:

    data/
      metadata_all.csv              <- satu tabel untuk semua speaker
      <speaker>/<kondisi>/<file>.wav   (16 kHz, mono, PCM 16-bit)
      _review/                      <- hal yang perlu dicek manusia

Menangani 4 bentuk folder yang ada di Drive kalian:
  1. Ada metadata.csv + WAV (S01, S02)            -> path Windows dibetulkan
  2. Ada metadata.csv + ZIP (S03)                 -> ZIP diekstrak dulu
  3. Hanya WAV bernama <spk>_<kondisi>_<id>.wav (S06) -> label diambil dari daftar induk
  4. Rekaman panjang "1-30.m4a", "31-40(bising).m4a" (S04)
                                                  -> dipotong otomatis berdasarkan jeda
  5. File bernama kalimatnya sendiri (S05)        -> set "spontan", label dari nama file

Pakai di Colab (folder bersama harus di-"Add shortcut to Drive" dulu):
    from google.colab import drive; drive.mount('/content/drive')
    !python prepare_dataset.py --src "/content/drive/MyDrive/Project KLP 5 - SR/Rekaman Data" \
        --out data --gender S06=P S04=L S05=P

Butuh: numpy, pandas, ffmpeg (sudah ada di Colab).
"""
import argparse
import re
import shutil
import subprocess
import sys
import wave
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

SR = 16000
AUDIO_EXT = {".wav", ".m4a", ".mp3", ".ogg", ".opus", ".webm", ".aac", ".flac"}

# ---------------------------------------------------------------- label ----
# Perintah 118-133 di metadata asli tidak punya piece/src/dst. Diisi manual di sini
# supaya kunci jawaban TIDAK bergantung pada parser yang sedang diuji.
EXTRA_GT = {
    118: ("kuda", "", "F3", ""),   119: ("pion", "", "E4", ""),
    120: ("gajah", "", "C4", ""),  121: ("benteng", "", "D1", ""),
    122: ("menteri", "", "H5", ""), 123: ("raja", "", "G1", ""),
    124: ("pion", "E7", "E8", "menteri"), 125: ("pion", "A7", "A8", "benteng"),
    126: ("pion", "H2", "H1", "kuda"),    127: ("pion", "B7", "B8", "gajah"),
    128: ("pion", "D4", "E5", ""), 129: ("menteri", "D1", "H5", ""),
    130: ("gajah", "C4", "F7", ""), 131: ("pion", "E2", "E4", ""),
    132: ("kuda", "G1", "F3", ""), 133: ("pion", "D2", "D4", ""),
}
PIECE_WORDS = {"pion", "kuda", "gajah", "benteng", "menteri", "ratu", "raja"}
CONTROL = [  # urutan penting: "batalkan langkah" dicek sebelum "batal"
    (r"\bbatalkan langkah\b|\bundo\b|\bmundur\b", "undo"),
    (r"\bbatal\w*\b", "cancel"),
    (r"\bulang\w*\b", "repeat"),
    (r"\btidak\b|\bbukan\b|\bnggak\b|\bgak\b", "no"),
    (r"\bya\b|\biya\b|\bbetul\b|\boke\b|\blanjut\w*\b", "yes"),
]
COLUMNS = ["id", "speaker", "gender", "condition", "set", "file", "reference", "type",
           "template_id", "expected_kind", "piece", "src", "dst", "promotion",
           "castle_side", "source", "needs_review", "note"]


def expected_from_text(text):
    """Kunci jawaban sederhana dari kalimat referensi (dipakai utk set spontan).
    Sengaja TIDAK memakai voicechess_parser agar evaluasinya tidak sirkular."""
    t = text.lower()
    squares = re.findall(r"\b([a-h])\s*-?\s*([1-8])\b", t)
    squares = [f"{c}{n}".upper() for c, n in squares]
    if "rokade" in t:
        side = "short" if "pendek" in t else "long" if "panjang" in t else ""
        return dict(expected_kind="castle", castle_side=side)
    if not squares:
        for pat, kind in CONTROL:
            if re.search(pat, t):
                return dict(expected_kind=kind)
        return dict(expected_kind="unknown")
    words = re.findall(r"[a-z]+", t)
    promo_at = next((i for i, w in enumerate(words) if w in ("jadi", "menjadi", "promosi")), None)
    head = words if promo_at is None else words[:promo_at]
    tail = [] if promo_at is None else words[promo_at + 1:]
    piece = next((w for w in head if w in PIECE_WORDS), "")
    promo = next((w for w in tail if w in PIECE_WORDS), "")
    src, dst = (squares[0], squares[1]) if len(squares) >= 2 else ("", squares[0])
    return dict(expected_kind="move", piece=piece, src=src, dst=dst, promotion=promo)


def plausibility_note(piece, src):
    """Tandai label yang mencurigakan (mis. 'pion dari B1': pion tak pernah di baris 1/8)."""
    if piece == "pion" and src and src[1] in "18":
        return "pion_di_baris_1/8?_cek_label"
    return ""


def load_master(src_root):
    """Ambil daftar 134 perintah dari metadata.csv mana pun yang lengkap (kondisi tenang)."""
    for csv in sorted(src_root.rglob("metadata.csv")):
        df = read_meta(csv)
        t = df[df.condition.str.lower() == "tenang"]
        if t.id.nunique() >= 134:
            m = t.drop_duplicates("id").set_index("id")
            print(f"[master] daftar perintah diambil dari {csv.relative_to(src_root)}")
            return m[["reference", "type", "template_id", "piece", "src", "dst"]]
    sys.exit("Tidak ada metadata.csv dengan 134 perintah 'tenang' untuk dijadikan daftar induk.")


def read_meta(path):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    df.columns = [c.strip().replace("\\_", "_") for c in df.columns]
    df["id"] = df["id"].astype(int)
    return df


def label_row(master, cid):
    r = master.loc[cid]
    piece, src, dst, promo = r.piece, r.src, r.dst, ""
    if cid in EXTRA_GT:
        piece, src, dst, promo = EXTRA_GT[cid]
    exp = expected_from_text(r.reference)
    return dict(reference=r.reference, type=r.type, template_id=r.template_id,
                expected_kind=exp["expected_kind"], castle_side=exp.get("castle_side", ""),
                piece=piece, src=src, dst=dst, promotion=promo,
                note=plausibility_note(piece, src))


# ---------------------------------------------------------------- audio ----
def decode(path):
    out = subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-i", str(path),
                          "-f", "s16le", "-ac", "1", "-ar", str(SR), "-"],
                         capture_output=True, check=True).stdout
    return np.frombuffer(out, np.int16)


def write_wav(path, pcm16):
    path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR)
        w.writeframes(pcm16.astype(np.int16).tobytes())


def to_standard_wav(src, dst):
    """Semua file diseragamkan ke 16 kHz mono 16-bit (WAV HP kadang 44.1/48 kHz stereo)."""
    pcm = decode(src)
    write_wav(dst, pcm)
    return len(pcm) / SR


def find_segments(pcm, delta_db, min_gap, min_dur=0.25, pad=0.15):
    """Potong berdasarkan energi: bagian di atas (lantai derau + delta_db) dianggap suara,
    jeda < min_gap detik digabung (jeda antar-kata), potongan < min_dur dibuang."""
    x = pcm.astype(np.float32) / 32768
    hop, win = int(0.01 * SR), int(0.03 * SR)
    n = max(1, 1 + (len(x) - win) // hop)
    frames = np.lib.stride_tricks.sliding_window_view(x, win)[::hop][:n]
    db = 10 * np.log10(np.mean(frames ** 2, axis=1) + 1e-10)
    floor = np.percentile(db, 10)
    voiced = db > floor + delta_db
    segs, start = [], None
    for i, v in enumerate(voiced):
        if v and start is None:
            start = i
        elif not v and start is not None:
            segs.append([start, i]); start = None
    if start is not None:
        segs.append([start, len(voiced)])
    merged = []
    for s in segs:
        if merged and (s[0] - merged[-1][1]) * 0.01 < min_gap:
            merged[-1][1] = s[1]
        else:
            merged.append(s)
    out = []
    for s, e in merged:
        if (e - s) * 0.01 >= min_dur:
            a = max(0, int((s * 0.01 - pad) * SR)); b = min(len(pcm), int((e * 0.01 + pad) * SR))
            out.append((a, b))
    return out


def split_to_n(pcm, n_expected):
    """Cari pengaturan yang menghasilkan tepat n potongan. Dari semua pengaturan yang
    cocok, pilih yang paling 'tengah' (paling stabil). Return (segmen, info, cocok?)."""
    hits, best = [], None
    for delta in (6, 8, 10, 12, 14, 16, 18, 20):
        for gap in (0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2, 1.5):
            segs = find_segments(pcm, delta, gap)
            diff = abs(len(segs) - n_expected)
            if diff == 0:
                hits.append((delta, gap, segs))
            if best is None or diff < best[0]:
                best = (diff, delta, gap, segs)
    if hits:
        delta, gap, segs = hits[len(hits) // 2]
        return segs, f"delta={delta}dB gap={gap}s ({len(hits)} pengaturan cocok)", True
    _, delta, gap, segs = best
    return segs, f"terdekat: {len(segs)} potongan (delta={delta}dB gap={gap}s)", False


# ------------------------------------------------------------- handlers ----
RANGE_RE = re.compile(r"^(\d+)\s*-\s*(\d+)\s*(?:\((\w+)\))?$", re.I)
NUMBERED_RE = re.compile(r"^(.+?)_([a-z]+)_(\d{3,4})$", re.I)


def handle_with_metadata(spk_dir, metas, out, master, unzip_root):
    rows, problems = [], []
    for z in spk_dir.rglob("*.zip"):  # S03: tenang.zip, jauh.zip
        dest = unzip_root / spk_dir.name / z.stem
        if not dest.exists():
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dest)
    index = {}
    for root in (spk_dir, unzip_root / spk_dir.name):
        if root.exists():
            for p in root.rglob("*"):
                if p.suffix.lower() in AUDIO_EXT:
                    index.setdefault(p.name.lower(), p)
    used = set()
    for csv in metas:
        for _, m in read_meta(csv).iterrows():
            base = m.file.replace("\\\\", "/").replace("\\", "/").split("/")[-1]
            src = index.get(base.lower())
            if src is None:
                problems.append(f"{spk_dir.name}: file di metadata tidak ditemukan: {base}")
                continue
            used.add(src.name.lower())
            dst = out / m.speaker / m.condition / Path(base).with_suffix(".wav").name
            dur = to_standard_wav(src, dst)
            lab = label_row(master, int(m.id))
            if lab["reference"].strip().lower() != m.reference.strip().lower():
                problems.append(f"{base}: reference beda dgn daftar induk "
                                f"('{m.reference}' vs '{lab['reference']}')")
            rows.append(dict(lab, id=int(m.id), speaker=m.speaker, gender=m.gender,
                             condition=m.condition, set="terbaca",
                             file=dst.relative_to(out).as_posix(), source=str(src.name),
                             needs_review=0, duration_s=round(dur, 2)))
    extra = sorted(set(index) - used)
    if extra:
        problems.append(f"{spk_dir.name}: {len(extra)} audio tidak tercatat di metadata, "
                        f"mis. {extra[:3]}")
    return rows, problems


def handle_numbered(spk_dir, files, out, master, genders):
    rows, problems = [], []
    for f in files:
        spk, cond, cid = NUMBERED_RE.match(f.stem).groups()
        cid = int(cid)
        if cid not in master.index:
            problems.append(f"{f.name}: id {cid} tidak ada di daftar induk"); continue
        dst = out / spk / cond / f.with_suffix(".wav").name
        dur = to_standard_wav(f, dst)
        rows.append(dict(label_row(master, cid), id=cid, speaker=spk,
                         gender=genders.get(spk, genders.get(spk_dir.name, "?")),
                         condition=cond, set="terbaca", file=dst.relative_to(out).as_posix(),
                         source=f.name, needs_review=0, duration_s=round(dur, 2)))
    ids = sorted({r["id"] for r in rows if r["condition"] == "tenang"})
    missing = sorted(set(range(134)) - set(ids))
    if ids and missing:
        problems.append(f"{spk_dir.name}: kondisi tenang kurang {len(missing)} id, mis. {missing[:10]}")
    return rows, problems


def handle_long(spk_dir, files, out, master, genders, review):
    rows, problems = [], []
    spk = spk_dir.name
    for f in sorted(files, key=lambda p: int(RANGE_RE.match(p.stem).group(1))):
        a, b, cond = RANGE_RE.match(f.stem).groups()
        cond = (cond or "tenang").lower()
        ids = list(range(int(a) - 1, int(b)))       # nama file 1-based -> id 0-based
        pcm = decode(f)
        segs, info, ok = split_to_n(pcm, len(ids))
        tag = f"{spk}/{f.name}: {len(ids)} perintah, {info}"
        if not ok:
            problems.append("PERINGATAN " + tag + " -> TIDAK dimasukkan; cek folder _review")
            for k, (s, e) in enumerate(segs):
                write_wav(review / spk / f.stem / f"potongan_{k:02d}.wav", pcm[s:e])
            pd.DataFrame([dict(potongan=k, mulai_s=round(s / SR, 2), selesai_s=round(e / SR, 2))
                          for k, (s, e) in enumerate(segs)]).to_csv(
                review / spk / f.stem / "potongan.csv", index=False)
            continue
        print("  [potong] " + tag)
        for cid, (s, e) in zip(ids, segs):
            name = f"{spk}_{cond}_{cid:04d}.wav"
            dst = out / spk / cond / name
            write_wav(dst, pcm[s:e])
            rows.append(dict(label_row(master, cid), id=cid, speaker=spk,
                             gender=genders.get(spk, "?"), condition=cond, set="terbaca",
                             file=dst.relative_to(out).as_posix(),
                             source=f"{f.name}@{s / SR:.2f}-{e / SR:.2f}s",
                             needs_review=1,  # tetap dengarkan acak beberapa potongan
                             duration_s=round((e - s) / SR, 2)))
    return rows, problems


def handle_spontan(spk_dir, files, out, genders):
    rows, problems = [], []
    spk = spk_dir.name
    for k, f in enumerate(sorted(files)):
        ref = f.stem.strip()
        exp = expected_from_text(ref)
        dst = out / spk / "spontan" / f"{spk}_spontan_{k:04d}.wav"
        dur = to_standard_wav(f, dst)
        note = plausibility_note(exp.get("piece", ""), exp.get("src", ""))
        if exp["expected_kind"] == "undo":
            note = "undo_belum_didukung_parser"
        rows.append(dict(exp, id=k, speaker=spk, gender=genders.get(spk, "?"),
                         condition="tenang", set="spontan", file=dst.relative_to(out).as_posix(),
                         reference=ref, type="spontan", template_id="", source=f.name,
                         needs_review=1, note=note, duration_s=round(dur, 2)))
    return rows, problems


# ----------------------------------------------------------------- main ----
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, help='folder "Rekaman Data"')
    ap.add_argument("--out", default="data")
    ap.add_argument("--gender", nargs="*", default=[], help="mis. S06=P S04=L")
    ap.add_argument("--skip", nargs="*", default=["Commands"], help="subfolder yang dilewati")
    args = ap.parse_args()

    src, out = Path(args.src), Path(args.out)
    review, unzip_root = out / "_review", out / "_unzip"
    genders = dict(g.split("=", 1) for g in args.gender)
    master = load_master(src)

    rows, problems = [], []
    for spk_dir in sorted(p for p in src.iterdir() if p.is_dir() and p.name not in args.skip):
        audio = [p for p in spk_dir.rglob("*") if p.suffix.lower() in AUDIO_EXT]
        metas = list(spk_dir.rglob("metadata.csv"))
        has_zip = any(spk_dir.rglob("*.zip"))
        print(f"[{spk_dir.name}] {len(audio)} audio, {len(metas)} metadata.csv, zip={has_zip}")
        if metas:
            r, p = handle_with_metadata(spk_dir, metas, out, master, unzip_root)
        elif audio and all(NUMBERED_RE.match(a.stem) for a in audio):
            r, p = handle_numbered(spk_dir, audio, out, master, genders)
        elif audio and all(RANGE_RE.match(a.stem) for a in audio):
            r, p = handle_long(spk_dir, audio, out, master, genders, review)
        elif audio:
            r, p = handle_spontan(spk_dir, audio, out, genders)
        else:
            r, p = [], [f"{spk_dir.name}: tidak ada audio"]
        rows += r; problems += p

    df = pd.DataFrame(rows)
    for c in COLUMNS:
        if c not in df:
            df[c] = ""
    df = df[COLUMNS + ["duration_s"]]
    df[COLUMNS] = df[COLUMNS].fillna("")
    out.mkdir(parents=True, exist_ok=True)
    df.to_csv(out / "metadata_all.csv", index=False)
    review.mkdir(parents=True, exist_ok=True)
    df[(df.needs_review == 1) | (df.note != "")].to_csv(review / "perlu_dicek.csv", index=False)

    print("\n=== Ringkasan (jumlah file per speaker x kondisi) ===")
    print(pd.crosstab([df.speaker, df.gender], [df.set, df.condition], margins=True))
    print("\nDurasi (detik):", df.duration_s.describe()[["min", "50%", "max"]].round(2).to_dict())
    short = df[df.duration_s < 0.4]
    if len(short):
        print(f"PERINGATAN: {len(short)} file < 0.4 dtk (mungkin kosong/terpotong):",
              list(short.file[:5]))
    print(f"\n=== {len(problems)} catatan ===")
    for p in problems:
        print(" -", p)
    print(f"\nTersimpan: {out / 'metadata_all.csv'} ({len(df)} baris)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval_dataset.py
"""
VoiceChess-ID - Evaluasi batch: WER, command accuracy, penanganan ragu, latency,
dan analisis salah dengar. Pengganti evaluasi() lama di voicechess_asr.ipynb.

Perbedaan penting dengan evaluasi() lama:
  * Kunci jawaban diambil dari kolom metadata (piece/src/dst/promotion), BUKAN dari
    parse(reference). Versi lama sirkular: kalau parser salah membaca referensi,
    kunci jawabannya ikut salah dan akurasinya tampak bagus.
  * Hasil Whisper disimpan ke results/asr_<model>_<prompt>.csv, jadi kalau Colab
    terputus, jalankan ulang dan ia melanjutkan (tidak mengulang dari nol).
  * Ada "pemanasan" per model (panggilan pertama tidak dihitung latency-nya).
  * Hasil dipisah menjadi 4 kategori keselamatan (lihat outcome()).

Colab (GPU T4 dianjurkan untuk batch; latency final tetap diukur di laptop demo):
    !pip install -q faster-whisper
    !python eval_dataset.py --meta data/metadata_all.csv --root data \
        --models tiny base small --prompt both --device cuda
Uji pipa tanpa Whisper (hipotesis = referensi, jadi WER=0; akurasi = batas atas parser):
    !python eval_dataset.py --meta data/metadata_all.csv --root data --dry-run
"""
import argparse
import difflib
import subprocess
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from voicechess_parser import normalize_for_wer, parse

PROMPT = ("Perintah catur: pion, kuda, gajah, benteng, menteri, raja, rokade. "
          "Contoh: Pion E2 ke E4. Kuda G1 ke F3.")
PIECE_EN = {"pion": "pawn", "kuda": "knight", "gajah": "bishop", "benteng": "rook",
            "menteri": "queen", "ratu": "queen", "raja": "king", "": None}


# ------------------------------------------------------------------ ASR ----
def load_audio(path, sr=16000):
    out = subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-i", str(path),
                          "-f", "f32le", "-ac", "1", "-ar", str(sr), "-"],
                         capture_output=True, check=True).stdout
    return np.frombuffer(out, np.float32)


_models = {}


def transcribe(path, model, prompt, device, compute_type):
    from faster_whisper import WhisperModel
    key = (model, device, compute_type)
    if key not in _models:
        _models[key] = WhisperModel(model, device=device, compute_type=compute_type)
    audio = load_audio(path)
    t = time.perf_counter()
    segs, _ = _models[key].transcribe(audio, language="id", beam_size=5, vad_filter=True,
                                      initial_prompt=prompt, condition_on_previous_text=False)
    text = " ".join(s.text.strip() for s in segs).strip()
    return text, time.perf_counter() - t


def run_asr(df, root, model, prompt, device, compute_type, cache):
    """Transkripsi semua file; hasil di-cache per baris supaya bisa dilanjutkan."""
    done = pd.read_csv(cache, keep_default_na=False) if cache.exists() else \
        pd.DataFrame(columns=["file", "hyp", "latency_s"])
    have = set(done.file)
    todo = [f for f in df.file.unique() if f not in have]
    if todo:
        transcribe(Path(root) / todo[0], model, prompt, device, compute_type)  # pemanasan
    new = []
    for i, f in enumerate(todo, 1):
        hyp, dt = transcribe(Path(root) / f, model, prompt, device, compute_type)
        new.append({"file": f, "hyp": hyp, "latency_s": round(dt, 3)})
        if i % 25 == 0 or i == len(todo):
            done = pd.concat([done, pd.DataFrame(new)], ignore_index=True)
            done.to_csv(cache, index=False); new = []
            print(f"    {model}/{'prompt' if prompt else 'tanpa'}: {len(done)}/{df.file.nunique()}")
    return done


# --------------------------------------------------------------- skoring ----
def gt_key(row):
    kind = row.expected_kind
    if kind == "move":
        return ("move", PIECE_EN.get(row.piece.lower()), row.src.lower() or None,
                row.dst.lower() or None, PIECE_EN.get(row.promotion.lower()))
    if kind == "castle":
        return ("castle", row.castle_side or None)
    return (kind,)


def pred_key(r):
    if r.kind == "move":
        return ("move", r.piece, r.src, r.dst, r.promotion)
    if r.kind == "castle":
        return ("castle", r.castle_side)
    return (r.kind,)


def outcome(correct, asked):
    """Kategori untuk bagian 'Error & Ambiguity Handling' di laporan."""
    if correct and not asked:
        return "1_benar_langsung"
    if correct and asked:
        return "2_benar_via_konfirmasi"
    if asked:
        return "3_salah_tapi_ditanya"      # aman: user bisa bilang 'tidak'
    return "4_salah_dieksekusi"            # berbahaya: langkah salah langsung jalan


def score(df, hyp_col="hyp"):
    rows = []
    for _, row in df.iterrows():
        r = parse(row[hyp_col])
        g, p = gt_key(row), pred_key(r)
        correct = g == p
        asked = r.needs_confirmation or r.kind == "unknown"
        comp = {}
        if row.expected_kind == "move":
            comp = dict(piece_ok=p[1:2] == g[1:2] if r.kind == "move" else False,
                        src_ok=r.kind == "move" and p[2] == g[2],
                        dst_ok=r.kind == "move" and p[3] == g[3])
        rows.append(dict(ref_norm=normalize_for_wer(row.reference),
                         hyp_norm=normalize_for_wer(row[hyp_col]),
                         pred=str(p), gold=str(g), cmd_ok=correct,
                         outcome=outcome(correct, asked), **comp))
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)


def wer(refs, hyps):
    """WER tingkat korpus (sama dengan jiwer.wer): total edit / total kata referensi."""
    err = tot = 0
    for r, h in zip(refs, hyps):
        a, b = r.split(), h.split()
        d = list(range(len(b) + 1))
        for i, x in enumerate(a, 1):
            prev, d[0] = d[0], i
            for j, y in enumerate(b, 1):
                prev, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, prev + (x != y))
        err += d[len(b)]; tot += len(a)
    return err / max(tot, 1)


def summarize(res, by):
    def agg(g):
        mv = g[g.expected_kind == "move"]
        out = {"n": len(g),
               "WER": wer(g.ref_norm, g.hyp_norm),
               "cmd_acc": g.cmd_ok.mean(),
               "piece_acc": mv.piece_ok.mean() if len(mv) else np.nan,
               "src_acc": mv.src_ok.mean() if len(mv) else np.nan,
               "dst_acc": mv.dst_ok.mean() if len(mv) else np.nan}
        if "latency_s" in g:
            out["lat_med_s"] = g.latency_s.median()
            out["lat_p90_s"] = g.latency_s.quantile(0.9)
        for k in ("1_benar_langsung", "2_benar_via_konfirmasi",
                  "3_salah_tapi_ditanya", "4_salah_dieksekusi"):
            out[k] = (g.outcome == k).mean()
        return pd.Series(out)
    return res.groupby(by).apply(agg, include_groups=False).round(3)


def confusions(res, top=25):
    """Pasangan salah dengar dari perataan token (referensi -> hipotesis)."""
    c = Counter()
    for ref, hyp in zip(res.ref_norm, res.hyp_norm):
        a, b = ref.split(), hyp.split()
        for op, i1, i2, j1, j2 in difflib.SequenceMatcher(a=a, b=b, autojunk=False).get_opcodes():
            if op == "replace" and i2 - i1 == j2 - j1:
                for x, y in zip(a[i1:i2], b[j1:j2]):
                    c[(x, y)] += 1
            elif op != "equal":
                c[(" ".join(a[i1:i2]) or "∅", " ".join(b[j1:j2]) or "∅")] += 1
    return pd.DataFrame([dict(referensi=k[0], terdengar=k[1], jumlah=v)
                         for k, v in c.most_common(top)])


# ----------------------------------------------------------------- main ----
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--meta", default="data/metadata_all.csv")
    ap.add_argument("--root", default="data")
    ap.add_argument("--out", default="results")
    ap.add_argument("--models", nargs="+", default=["small"])
    ap.add_argument("--prompt", choices=["none", "yes", "both"], default="both")
    ap.add_argument("--device", default="cpu")
    ap.add_argument("--compute-type", default=None, help="default: int8 (cpu) / float16 (cuda)")
    ap.add_argument("--speakers", nargs="*", help="hanya speaker ini (mis. set tuning)")
    ap.add_argument("--set", dest="subset", default=None, help="terbaca / spontan")
    ap.add_argument("--dry-run", action="store_true", help="hipotesis = referensi")
    a = ap.parse_args()
    ct = a.compute_type or ("float16" if a.device == "cuda" else "int8")

    df = pd.read_csv(a.meta, dtype=str, keep_default_na=False)
    if a.speakers:
        df = df[df.speaker.isin(a.speakers)]
    if a.subset:
        df = df[df.set == a.subset]
    out = Path(a.out); out.mkdir(exist_ok=True)

    # Batas atas parser: parser dijalankan pada teks referensi yang sempurna.
    up = score(df.assign(hyp=df.reference))
    print("\n=== Parser pada teks referensi (tanpa ASR) ===")
    print(summarize(up, ["set", "type"])[["n", "cmd_acc", "4_salah_dieksekusi"]])
    bad = up[~up.cmd_ok]
    if len(bad):
        print("\nParser salah membaca referensi (perbaiki kamus/aturan, atau labelnya):")
        print(bad[["speaker", "reference", "gold", "pred"]].drop_duplicates("reference").to_string())
    if a.dry_run:
        return

    configs = [(m, p) for m in a.models
               for p in ([None, PROMPT] if a.prompt == "both" else
                         [PROMPT] if a.prompt == "yes" else [None])]
    allres = []
    for m, p in configs:
        tag = f"{m}_{'prompt' if p else 'tanpa'}"
        print(f"\n--- {tag} ---")
        asr = run_asr(df, a.root, m, p, a.device, ct, out / f"asr_{tag}.csv")
        res = score(df.merge(asr, on="file", how="left").fillna({"hyp": ""}))
        res["model"], res["prompt"] = m, "ya" if p else "tidak"
        res["latency_s"] = pd.to_numeric(res.latency_s, errors="coerce")
        res.to_csv(out / f"detail_{tag}.csv", index=False)
        allres.append(res)

    res = pd.concat(allres, ignore_index=True)
    pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
    tabel = summarize(res, ["model", "prompt", "set"])
    print("\n=== Ringkasan per model x prompt ===\n", tabel)
    tabel.to_csv(out / "ringkasan_model.csv")
    per = summarize(res, ["model", "prompt", "speaker", "condition"])
    per.to_csv(out / "ringkasan_speaker_kondisi.csv")
    print("\n(per speaker x kondisi disimpan di", out / "ringkasan_speaker_kondisi.csv", ")")

    # Analisis salah dengar: gabungan semua konfigurasi (paling banyak contoh error),
    # plus daftar per konfigurasi di CSV.
    per_cfg = []
    for (m, p), g in res.groupby(["model", "prompt"]):
        per_cfg.append(confusions(g, 15).assign(model=m, prompt=p))
    pd.concat(per_cfg).to_csv(out / "salah_dengar_per_konfigurasi.csv", index=False)
    conf = confusions(res, 30)
    conf.to_csv(out / "salah_dengar.csv", index=False)
    print("\n=== Salah dengar terbanyak (semua konfigurasi digabung) ===")
    print(conf.to_string() if len(conf) else "  (tidak ada)")
    for col in ("speaker", "condition"):
        print(f"\nSalah dengar per {col}:")
        for k, g in res.groupby(col):
            top = confusions(g, 6)
            print(f"  {k}: " + ("; ".join(f"{r.referensi}->{r.terdengar} ({r.jumlah})"
                                          for r in top.itertuples()) or "-"))
    worst = res[~res.cmd_ok][["model", "prompt", "speaker", "condition", "reference", "hyp",
                              "gold", "pred", "outcome"]]
    worst.to_csv(out / "perintah_gagal.csv", index=False)
    print(f"\n{len(worst)} perintah gagal -> {out / 'perintah_gagal.csv'}")

if __name__ == "__main__":
    main()


## 3. Parser
Unggah `voicechess_parser.py` kalian (versi terbaru). Sel ini dilewati otomatis kalau file sudah ada.

In [ ]:
import os
if not os.path.exists('voicechess_parser.py'):
    from google.colab import files
    files.upload()   # pilih voicechess_parser.py
!python voicechess_parser.py | tail -3

## 4. Satukan dataset
Gender S01, S02, S03 sudah ada di metadata mereka. Ganti `S04=L` di `GENDER` kalau salah.

In [ ]:
GENDER = "S06=P S05=P S04=L"   # ganti dengan nama folder speaker di Drive
!python prepare_dataset.py --src "$SRC" --out data --gender $GENDER

## 5. Cek yang perlu dicek manusia
**a.** Daftar baris yang ditandai (potongan S04, label S05).

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 200); pd.set_option('display.width', 200)
cek = pd.read_csv('data/_review/perlu_dicek.csv', dtype=str, keep_default_na=False)
print(cek.groupby(['speaker', 'condition']).size())
cek[cek.note != ''][['speaker', 'reference', 'note', 'source']]

**b.** Dengarkan potongan S04: pertama, tengah, terakhir dari tiap rekaman panjang. Pastikan yang terdengar sama dengan teks di atas pemutar. Kalau mulai bergeser, beri tahu (tempel nama filenya).

In [ ]:
SPK_LONG = 'S04'   # ganti dengan nama folder speaker di Drive
from IPython.display import Audio, display
meta = pd.read_csv('data/metadata_all.csv', dtype=str, keep_default_na=False)
d = meta[meta.speaker == SPK_LONG].copy()
d['rekaman'] = d.source.str.split('@').str[0]
for rek, g in d.groupby('rekaman'):
    g = g.sort_values('id', key=lambda s: s.astype(int))
    for _, r in g.iloc[[0, len(g) // 2, -1]].iterrows():
        print(f'{rek} | perintah {int(r.id) + 1}: {r.reference}')
        display(Audio(f'data/{r.file}'))

**c.** Kalau ada rekaman S04 yang ditolak (jumlah potongan tidak cocok), potongannya ada di `data/_review/<folder S04>/<nama>/`. Sel ini memutar semuanya supaya bisa dicari di mana yang salah.

In [ ]:
import glob
for folder in sorted(glob.glob('data/_review/*/*/')):
    print('==', folder)
    for f in sorted(glob.glob(folder + '*.wav')):
        print(os.path.basename(f)); display(Audio(f))

## 6. Dry-run: parser pada teks referensi (tanpa Whisper)
Ini batas atas: kalau parser sudah salah di teks yang sempurna, perbaiki parser/label dulu sebelum langkah 7.

In [ ]:
!python eval_dataset.py --meta data/metadata_all.csv --root data --dry-run

## 7. Evaluasi Whisper (lama)
tiny/base/small × dengan/tanpa prompt. Hasil tersimpan di `results/`; kalau Colab terputus, jalankan ulang sel ini dan ia melanjutkan.

Untuk menyimpan hasil ke Drive (supaya tidak hilang), ganti `--out results` dengan `--out "$SRC/../results"`.

In [ ]:
!pip install -q faster-whisper
!python eval_dataset.py --meta data/metadata_all.csv --root data --models tiny base small --prompt both --device cuda --out results

## 8. Lihat hasil

In [ ]:
display(pd.read_csv('results/ringkasan_model.csv'))
display(pd.read_csv('results/salah_dengar.csv').head(20))
display(pd.read_csv('results/ringkasan_speaker_kondisi.csv'))